In [1]:
# library imports
import pandas as pd 
import requests
import os 
import time 
import logging

In [10]:
# configure variable
log_file_path = "../logs"
log_file_name = "data_extraction.log"
data_file_path = "../data/raw"
currency = "usd" 
coins_per_page = 250
pages = 5
url = "https://api.coingecko.com/api/v3/coins/markets" 
sleep_time = 2 
max_retries = 3

# creating folder if not exists
os.makedirs(log_file_path, exist_ok=True)
os.makedirs(data_file_path, exist_ok=True) 

# setting logging parameters with initial message
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s", 
    handlers= [ 
        logging.FileHandler(os.path.join(log_file_path,log_file_name)),
        logging.StreamHandler() ]
)
logger = logging.getLogger(__name__)
logger.info("Data Extraction Started")

2026-06-20 05:35:29,271 - INFO - Data Extraction Started


In [11]:
# constant params
fixed_params = {
    "vs_currency" : currency,
    "per_page": coins_per_page,
    "order" : "market_cap_desc"
}
file_suffix = "_coinGecko.csv"

def get_response(page_number):
    # setting params
    params = fixed_params.copy()
    params["page"] = page_number
    response = requests.get(url, params=params)
    return response
    
def fetch_page(page_number):
    for i in range(1, max_retries+1):
        try:
            response = get_response(page_number)
            status_code = response.status_code
            data = response.json()
            len_data = len(data)
            if status_code == 200: 
                if len_data != 0:
                    df = pd.DataFrame(data)
                    df.to_csv(os.path.join(data_file_path,"page_"+str(page_number)+file_suffix), index = False)
                    logger.info(f"page_{page_number}{file_suffix} is created successfully") 
                    return df
                else:
                    if i > 2: 
                        logger.error(f"page_{page_number}{file_suffix} is not having data")
                        return None
                    else:
                        logger.info(f"No data fetched after {i} retry") 
                        time.sleep(sleep_time)
            elif status_code == 429:
                if i<4:
                    logger.warning(f"Error 429 : Too many requests\nTrying {i} time with sleep time {2**i}")
                    time.sleep(2**i)
                    continue
            else:
                logger.error(f"Execution stopped with status code : {status_code}")
                return None
        except Exception as e:
            logger.error(f"Execution stopped with error : {e}")
            return None
    else:
        logger.error(f"Max tries limit reached")
        return None

In [12]:
recovery_file = "file_state.txt"
def main():
    unfetched_pages = list(range(1, pages+1))
    all_data = []
    pages_fetched = []
    final_data = None
    if os.path.exists(os.path.join(log_file_path,recovery_file)): 
        logger.info("Retriving pages fetched")
        with open(os.path.join(log_file_path,recovery_file), "r") as f:
            pages_fetched = [line.strip() for line in f.readlines()]
                   
    if len(pages_fetched) > 0:
        for i in pages_fetched:
            if int(i) in unfetched_pages:
                unfetched_pages.remove(int(i))
    
    unfetched_pages.sort()
    for page_number in unfetched_pages:
        data_frame = fetch_page(page_number) 
        if data_frame is not None:
            all_data.append(data_frame)
            with open(os.path.join(log_file_path,recovery_file), "a") as f: 
                f.write(f"{page_number}\n")
            time.sleep(sleep_time)
        else: 
            logger.warning(f"Page {page_number} failed - skipping")
            
    all_data_len = len(all_data) 
    if all_data_len == 0:
        logger.error("No data extracted")
    else:
        if all_data_len == 1:
            final_data = all_data[0]
        else:
            final_data = pd.concat(all_data, ignore_index=True)
        
        final_data_rows= final_data.shape[0]
        final_data_columns = final_data.shape[1]
        if final_data_rows >= 500:
            final_data.to_csv(os.path.join(data_file_path,"raw_data.csv"), index = False)
            logger.info(f"Final raw_data.csv is created successfully") 
        else: 
            logger.error("Insufficient Data for analysis")

        if all_data_len == pages:
            if os.path.exists(os.path.join(log_file_path,recovery_file)):
                os.remove(os.path.join(log_file_path,recovery_file)) 
                logger.info("Recovery file deleted- Pipeline complete")
        logger.info(f"Total rows: {final_data_rows}") 
        logger.info(f"Total columns : {final_data_columns}")
        logger.info(f"Data Extraction Completed")
        print(final_data.head())

In [13]:
main()

2026-06-20 05:35:33,963 - INFO - page_1_coinGecko.csv is created successfully
2026-06-20 05:35:36,324 - INFO - page_2_coinGecko.csv is created successfully
2026-06-20 05:35:38,663 - INFO - page_3_coinGecko.csv is created successfully
2026-06-20 05:35:41,054 - INFO - page_4_coinGecko.csv is created successfully
2026-06-20 05:35:43,449 - INFO - page_5_coinGecko.csv is created successfully
2026-06-20 05:35:45,509 - INFO - Final raw_data.csv is created successfully
2026-06-20 05:35:45,509 - INFO - Recovery file deleted- Pipeline complete
2026-06-20 05:35:45,516 - INFO - Total rows: 1250
2026-06-20 05:35:45,518 - INFO - Total columns : 26
2026-06-20 05:35:45,519 - INFO - Data Extraction Completed


            id symbol      name  \
0      bitcoin    btc   Bitcoin   
1     ethereum    eth  Ethereum   
2       tether   usdt    Tether   
3  binancecoin    bnb       BNB   
4     usd-coin   usdc      USDC   

                                               image  current_price  \
0  https://coin-images.coingecko.com/coins/images...   63522.000000   
1  https://coin-images.coingecko.com/coins/images...    1709.630000   
2  https://coin-images.coingecko.com/coins/images...       0.999123   
3  https://coin-images.coingecko.com/coins/images...     581.110000   
4  https://coin-images.coingecko.com/coins/images...       0.999818   

      market_cap  market_cap_rank  fully_diluted_valuation  total_volume  \
0  1273141961890                1            1273141961890  2.139618e+10   
1   206385735727                2             206385735727  8.027663e+09   
2   186259989549                3             191721750256  3.810132e+10   
3    78318302876                4              78318302876